In [0]:
%sql
-- ============================================================
-- Insert 2 new records + update existing records in bronze tables
-- Tables: product_catalog, inventory (retail_q.postgres_bronze), account, opportunity (retail_q.salesforce_bronze)
-- ============================================================

-- --------------------------------------------------------------
-- 1. product_catalog — INSERT 2 new records
-- --------------------------------------------------------------
MERGE INTO retail_q.postgres_bronze.product_catalog AS t
USING (
    SELECT 'PROD-9001' AS product_id, 'Wireless Noise-Cancelling Headphones' AS product_name, 'Electronics' AS category,
           CAST(199.99 AS DECIMAL(10,2)) AS unit_price, true AS is_active, current_date() AS launch_date, current_timestamp() AS updated_at
    UNION ALL
    SELECT 'PROD-9002' AS product_id, 'Smart Home Hub Gen 3' AS product_name, 'Electronics' AS category,
           CAST(89.99 AS DECIMAL(10,2)) AS unit_price, true AS is_active, current_date() AS launch_date, current_timestamp() AS updated_at
) AS s
ON t.product_id = s.product_id
WHEN NOT MATCHED THEN INSERT (product_id, product_name, category, unit_price, is_active, launch_date, updated_at)
VALUES (s.product_id, s.product_name, s.category, s.unit_price, s.is_active, s.launch_date, s.updated_at);

-- product_catalog — UPDATE existing records
MERGE INTO retail_q.postgres_bronze.product_catalog AS t
USING (SELECT 'P0001' AS product_id, CAST(249.99 AS DECIMAL(10,2)) AS unit_price, current_timestamp() AS updated_at) AS s
ON t.product_id = s.product_id
WHEN MATCHED THEN UPDATE SET unit_price = s.unit_price, updated_at = s.updated_at;

MERGE INTO retail_q.postgres_bronze.product_catalog AS t
USING (SELECT 'P0002' AS product_id, false AS is_active, current_timestamp() AS updated_at) AS s
ON t.product_id = s.product_id
WHEN MATCHED THEN UPDATE SET is_active = s.is_active, updated_at = s.updated_at;

-- --------------------------------------------------------------
-- 2. inventory — INSERT 2 new records
-- --------------------------------------------------------------
MERGE INTO retail_q.postgres_bronze.inventory AS t
USING (
    SELECT 'INV-9001' AS inventory_id, 'PROD-9001' AS product_id, 'WH-01' AS store_id,
           500 AS stock_quantity, 100 AS reorder_level, current_timestamp() AS last_stock_update
    UNION ALL
    SELECT 'INV-9002' AS inventory_id, 'PROD-9002' AS product_id, 'WH-01' AS store_id,
           320 AS stock_quantity, 80 AS reorder_level, current_timestamp() AS last_stock_update
) AS s
ON t.inventory_id = s.inventory_id
WHEN NOT MATCHED THEN INSERT (inventory_id, product_id, store_id, stock_quantity, reorder_level, last_stock_update)
VALUES (s.inventory_id, s.product_id, s.store_id, s.stock_quantity, s.reorder_level, s.last_stock_update);

-- inventory — UPDATE existing records
MERGE INTO retail_q.postgres_bronze.inventory AS t
USING (SELECT 'INV0001' AS inventory_id, current_timestamp() AS last_stock_update) AS s
ON t.inventory_id = s.inventory_id
WHEN MATCHED THEN UPDATE SET stock_quantity = stock_quantity + 200, last_stock_update = s.last_stock_update;

MERGE INTO retail_q.postgres_bronze.inventory AS t
USING (SELECT 'INV0002' AS inventory_id, 150 AS reorder_level, current_timestamp() AS last_stock_update) AS s
ON t.inventory_id = s.inventory_id
WHEN MATCHED THEN UPDATE SET reorder_level = s.reorder_level, last_stock_update = s.last_stock_update;

-- --------------------------------------------------------------
-- 3. account — INSERT 2 new records
-- --------------------------------------------------------------
MERGE INTO retail_q.salesforce_bronze.account AS t
USING (
    SELECT 'ACC-9001' AS Id, 'Acme Robotics Inc.' AS Name, 'Manufacturing' AS Industry,
           'North America' AS BillingCountry, '123 Innovation Dr, Austin TX' AS BillingStreet,
           '512-555-0188' AS Phone, 'www.acmerobotics.com' AS Website,
           current_timestamp() AS CreatedDate, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp
    UNION ALL
    SELECT 'ACC-9002' AS Id, 'BlueSky Analytics LLC' AS Name, 'Technology' AS Industry,
           'Europe' AS BillingCountry, '45 Rue du Commerce, Paris' AS BillingStreet,
           '+33-1-555-0142' AS Phone, 'www.blueskyanalytics.io' AS Website,
           current_timestamp() AS CreatedDate, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp
) AS s
ON t.Id = s.Id
WHEN NOT MATCHED THEN INSERT (Id, Name, Industry, BillingCountry, BillingStreet, Phone, Website, CreatedDate, LastModifiedDate, SystemModstamp)
VALUES (s.Id, s.Name, s.Industry, s.BillingCountry, s.BillingStreet, s.Phone, s.Website, s.CreatedDate, s.LastModifiedDate, s.SystemModstamp);

-- account — UPDATE existing records
MERGE INTO retail_q.salesforce_bronze.account AS t
USING (SELECT '001f600000g5UmtAAE' AS Id, 'APAC' AS BillingCountry, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp) AS s
ON t.Id = s.Id
WHEN MATCHED THEN UPDATE SET BillingCountry = s.BillingCountry, LastModifiedDate = s.LastModifiedDate, SystemModstamp = s.SystemModstamp;

MERGE INTO retail_q.salesforce_bronze.account AS t
USING (SELECT '001f600000g5UmuAAE' AS Id, '+1-415-555-0199' AS Phone, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp) AS s
ON t.Id = s.Id
WHEN MATCHED THEN UPDATE SET Phone = s.Phone, LastModifiedDate = s.LastModifiedDate, SystemModstamp = s.SystemModstamp;

-- --------------------------------------------------------------
-- 4. opportunity — INSERT 2 new records
-- --------------------------------------------------------------
MERGE INTO retail_q.salesforce_bronze.opportunity AS t
USING (
    SELECT 'OPP-9001' AS Id, 'ACC-9001' AS AccountId, 'Acme – Robotics Platform Deal' AS Name,
           'Qualification' AS StageName, CAST(250000.00 AS DECIMAL(38,2)) AS Amount,
           DATE '2025-09-30' AS CloseDate, 'USER-201' AS OwnerId, 0.25 AS Probability,
           current_timestamp() AS CreatedDate, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp
    UNION ALL
    SELECT 'OPP-9002' AS Id, 'ACC-9002' AS AccountId, 'BlueSky – Analytics Pilot' AS Name,
           'Prospecting' AS StageName, CAST(120000.00 AS DECIMAL(38,2)) AS Amount,
           DATE '2025-10-15' AS CloseDate, 'USER-202' AS OwnerId, 0.10 AS Probability,
           current_timestamp() AS CreatedDate, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp
) AS s
ON t.Id = s.Id
WHEN NOT MATCHED THEN INSERT (Id, AccountId, Name, StageName, Amount, CloseDate, OwnerId, Probability, CreatedDate, LastModifiedDate, SystemModstamp)
VALUES (s.Id, s.AccountId, s.Name, s.StageName, s.Amount, s.CloseDate, s.OwnerId, s.Probability, s.CreatedDate, s.LastModifiedDate, s.SystemModstamp);

-- opportunity — UPDATE existing records
MERGE INTO retail_q.salesforce_bronze.opportunity AS t
USING (SELECT '006f600000BrU02AAF' AS Id, 'Negotiation' AS StageName, CAST(510000.00 AS DECIMAL(38,2)) AS Amount, 0.70 AS Probability, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp) AS s
ON t.Id = s.Id
WHEN MATCHED THEN UPDATE SET StageName = s.StageName, Amount = s.Amount, Probability = s.Probability, LastModifiedDate = s.LastModifiedDate, SystemModstamp = s.SystemModstamp;

MERGE INTO retail_q.salesforce_bronze.opportunity AS t
USING (SELECT '006f600000BrU03AAF' AS Id, 'Closed Won' AS StageName, 1.00 AS Probability, current_timestamp() AS LastModifiedDate, current_timestamp() AS SystemModstamp) AS s
ON t.Id = s.Id
WHEN MATCHED THEN UPDATE SET StageName = s.StageName, Probability = s.Probability, LastModifiedDate = s.LastModifiedDate, SystemModstamp = s.SystemModstamp;